# Explore TriviaQA → pick source questions (Week 1, P1)

**Purpose:** look at the data, check that the filters behave sensibly, and spot-check answers.

**Rule:** the filter rules live in `data/build_source_questions.py` (single source of truth).
This notebook *imports* them, so what you inspect here is exactly what the script exports.
Do **not** export `source_questions.json` from this notebook. Run the script instead:

```
python data/build_source_questions.py --n 800 --seed 42
```

Workflow: run cells 1-5 → read the questions → tune filters (override `cfg` below, or edit the
script; autoreload picks up edits) → run the script → come back for the answer spot-check at the end.

## 1. Setup

In [3]:
# !pip install datasets pandas
%load_ext autoreload
%autoreload 2

import sys, random
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR = Path.cwd().parent / "data"      # assumes this notebook lives in notebooks/
sys.path.insert(0, str(DATA_DIR))
import build_source_questions as bsq

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
print("Using data dir:", DATA_DIR)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using data dir: d:\data


## 2. Load the dataset and check its structure

In [4]:
CONFIG, SPLIT = "unfiltered.nocontext", "train"
records = list(bsq.load_triviaqa(CONFIG, SPLIT))
print(len(records), "rows")

87622 rows


In [5]:
# Confirm the field names BEFORE trusting the filters (they read answer["value"], ["aliases"], ["type"]).
r = records[0]
print("question:", r["question"])
print("answer keys:", list(r["answer"].keys()))
print({k: v for k, v in r["answer"].items() if k not in ("aliases", "normalized_aliases")})
print("number of aliases:", len(r["answer"].get("aliases", [])))
if "value" not in r["answer"]:
    print("\n!! 'value' is missing: the field names differ from what the script expects. Fix load_triviaqa().")

question: Who was President when the first Peanuts cartoon was published?
answer keys: ['aliases', 'normalized_aliases', 'matched_wiki_entity_name', 'normalized_matched_wiki_entity_name', 'normalized_value', 'type', 'value']
{'matched_wiki_entity_name': '', 'normalized_matched_wiki_entity_name': '', 'normalized_value': 'harry truman', 'type': 'WikipediaEntity', 'value': 'Harry Truman'}
number of aliases: 25


In [6]:
df = pd.DataFrame({
    "question":    [r["question"] for r in records],
    "answer":      [r["answer"].get("value", "") for r in records],
    "answer_type": [r["answer"].get("type") for r in records],
    "n_aliases":   [len(r["answer"].get("aliases", [])) for r in records],
    "source":      [r["question_source"] for r in records],
})
df["q_words"] = df.question.str.split().str.len()
df["a_words"] = df.answer.str.split().str.len()
df.head()

,question,answer,answer_type,n_aliases,source,q_words,a_words
0,Who was President when the first Peanuts cartoon was published?,Harry Truman,WikipediaEntity,25,http://www.triviacountry.com/,10,2
1,Which American-born Sinclair won the Nobel Prize for Literature in 1930?,Sinclair Lewis,WikipediaEntity,5,http://www.triviacountry.com/,11,2
2,Where in England was Dame Judi Dench born?,York,WikipediaEntity,21,http://www.triviacountry.com/,8,1
3,"William Christensen of Madison, New Jersey, has claimed to have the world's biggest collection of what?",Beer Cans,FreeForm,1,http://www.triviacountry.com/,16,2
4,In which decade did Billboard magazine first publish and American hit chart?,30s,WikipediaEntity,5,http://www.triviacountry.com/,12,1


## 3. Read the data (don't skip this)
Read ~50 random questions. Roughly how many are short? Unambiguous? Easy to modify
(year, number, entity, unit, comparative)? Write your impressions into the data-source note.

In [7]:
display(df.sample(50, random_state=0)[["question", "answer", "answer_type"]])

,question,answer,answer_type
24809,"What expression referring to admitting a mistake, humility and shame, derives from an old English peasant dish made ...",Eat humble pie,FreeForm
86077,"August 11, 1988 is generally accepted as the creation date of what terrorist organization, whose name translates as ...",Al Qaeda,WikipediaEntity
29176,In Rugby Union what position does the number 15 play?,Full-back,WikipediaEntity
15705,Which was the first capital city to host the Winter Olympics?,Oslo,WikipediaEntity
23210,Which European country found itself without a government from June 2010 into 2011 because of failed coalition talks ...,Belgium,WikipediaEntity
51414,Which Polynesian word means forbidden?,Taboo,WikipediaEntity
14874,Which country finally joined the United Nations in 2002?,Switzerland,WikipediaEntity
26576,"The famous tea brandname Typhoo, created c.1903 by the Sumner pharmacy/grocers in Birmingham UK, is from the Chinese...",Doctor,WikipediaEntity
37552,Australian Joan Sutherland is connected with what activity?,Opera,WikipediaEntity
76055,"In which country is the Tower of Hercules, the oldest roman lighthouse still in use today?",Spain,WikipediaEntity


In [8]:
print(df.q_words.describe(), "\n")
print(df.a_words.describe(), "\n")
print(df.answer_type.value_counts(dropna=False), "\n")
print(df.source.value_counts().head())

count    87622.000000
mean        13.988268
std          7.015856
min          5.000000
25%          9.000000
50%         12.000000
75%         17.000000
max        184.000000
Name: q_words, dtype: float64 

count    87622.000000
mean         1.964621
std          1.721361
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max        110.000000
Name: a_words, dtype: float64 

answer_type
WikipediaEntity    74616
FreeForm            9362
Numerical           3388
Range                256
Name: count, dtype: int64 

source
www.sfquiz.org.uk                20470
http://www.odquiz.org.uk/        11436
http://www.quizwise.com/         10994
http://www.quizballs.com/         7993
http://www.businessballs.com/     7139
Name: count, dtype: int64


## 4. Apply the filters and inspect what they keep and drop
`cfg` below uses the script defaults. To experiment, override a value, e.g.
`bsq.FilterConfig(max_words=15)`. If you settle on non-default values, pass the same
values as flags to the script (`--max-words 15`) so the export matches.

In [9]:
cfg = bsq.FilterConfig()
print(cfg)

results = [bsq.evaluate(r, cfg) for r in records]
df["reject_reason"] = [res[0] for res in results]
df["features"] = [",".join(res[1]) for res in results]

df["reject_reason"].fillna("KEPT").value_counts()

FilterConfig(min_words=4, max_words=20, max_answer_words=5)


reject_reason
KEPT                     62791
question_length          11198
no_modifiable_feature     9399
answer_too_long           2626
multi_answer               996
time_sensitive             612
Name: count, dtype: int64

In [10]:
# Kept questions: do they look like good source questions?
kept = df[df.reject_reason.isna()]
print(len(kept), "kept before dedupe/sampling")
display(kept.sample(30, random_state=1)[["question", "answer", "features"]])

62791 kept before dedupe/sampling


,question,answer,features
38694,Which country became independent from imperial China in 938 AD in the Battle of Bch ng River?,Vietnam,"number,entity"
45440,Who most famously had a horse name Bucephalus?,Alexander the Great,"comparative,entity"
62930,Which clothing retailer founded in 1975 is based in Arteixo in northern Spain?,ZARA,"year,entity"
20132,Which 1966 Beatles album shares its name with a weapon in Cluedo?,REVOLVER,"year,entity"
966,In 1984 how was the baby who received the heart of a baboon known?,Baby Fae,year
1279,How is Tenzin Gyatso better known?,The Dali Lama,entity
81402,During World War 2 which service was known as the 'Brylcreem Boys?,RAF,"number,entity"
46284,What type of product was sold under the brand name 'Abdulla'?,Cigarettes,entity
12423,"In the children’s nursery rhyme, where was ‘The Cock Horse’ ridden to?",Banbury Cross,entity
43847,What street do Bert and Ernie live on?,Sesame Street,entity


In [11]:
# Rejected questions: are the filters throwing away good ones, or are they right?
for reason in df.reject_reason.dropna().unique():
    sub = df[df.reject_reason == reason]
    print(f"\n== {reason}  ({len(sub)} rows)")
    display(sub.sample(min(6, len(sub)), random_state=2)[["question", "answer"]])


== answer_too_long  (2626 rows)


,question,answer
58947,Frenchman Robert Hubert was executed in England during the latter part of the 17th century after he falsely confesse...,Starting the Great Fire of London
68888,Fashion. What is a 'Wide-Awake'?,A soft felt HAT with a low crown and wide brim
3933,What are the names of the historic Cinque ports in Southern England?,"Dover, Hastings, New Romney, Hythe and Sandwich"
338,Vincent Van Gogh's painting Sunflowers was sold at auction for $39.9 million in 1987. How much did that come to per ...,$2.66 million. There are 15 sunflowers in the painting
5640,Who or what was Humpty Dumpty?,A cannon or field gun used in the siege of Colchester during the English Civil War
28384,What is the highest rank of officer in the RAF?,Marshall of the Royal Air Force



== multi_answer  (996 rows)


,question,answer
2603,The artist Leonardo Da Vinci was alive in which two centuries?,15th and 16th
66789,"In football, which two England central defenders of the 1980's shared the same forename and surname?",DAVE WATSON
57651,"Name the bandleader who died in March 2013, his biggest hits were 'Samantha' and 'Midnight In Moscow'?",Kenny Ball
69967,"Who is missing from this list - 'Dancer', 'Dasher', 'Donner' ,Blitzen', 'Vixen', 'Prancer' and 'Comet'?",'CUPID'
24464,"Name the mythological Greek hero King of Salamis, son of Telamon and Periboea, and cleaning product?",Ajax
16354,Name the country which ruled Burundi after Germany?,Belgium



== no_modifiable_feature  (9399 rows)


,question,answer
85503,"What can be a color (pinkish-orange), a marine animal, and a type of snake?",Coral
32217,Dermatitis affects what part of the body?,Skin
67782,"""Which brand of cigarette had the slogan, """"cool as a mountain stream""""?""",CONSULATE
41906,Camembert is a type of what?,Cheese
85158,"Which companies advertises it ""gets the red out""?",Visine
67562,How many valves are there on a (standard) trumpet?,THREE



== question_length  (11198 rows)


,question,answer
64695,"George Bailey, the character portrayed by James Stewart in It's a Wonderful Life is loosely based upon which banker ...",Amadeo Giannini (1870-1949)
65730,What classic short story by the French writer Guy de Maupassant concerns an invisible malevolent spirit that aims to...,The Horla
66685,"In the BBC Curse of comedy series, name either actor who played Wilfred Brambell or Harry H Corbett in the episode T...",Phil Davis or Jason Isaacs
50101,"During August 2009, in which country was the ruling party voted out of office after 54 years of virtually unbroken r...",Japan
32316,"What collection includes 3 King's crowns. 3 Queen's crowns, 5 sceptres, 2 orbs, St Edward's staff, 5 swords and 2 br...",Britain's Crown Jewels
35423,"From 1885 to 1908, a corporate state of 2,344,000 km (905,000 square miles), privately controlled by Leopold II, Kin...",Congo Free State



== time_sensitive  (612 rows)


,question,answer
67708,Which recent BBC TV drama set in Pagford opens with the death of a parish councillor?,THE CASUAL VACANCY
71471,Which instrument measures the electrical current in a circuit?,AMMETER
75730,"To date only 3 teams have won the Women’s Rugby World Cup, England, New Zealand and which other nation’s?",U.S.A.
19691,"Which country are the reigning Olympic Rugby Union champions, the sport last being part of the official programme in...",USA
21047,"Born in Llwynpia in South Wales in 1971, who is the current leader of Plaid Cymru ?",(Leanne) WOODS
68388,Who is the current monarch of Belgium?,KING ALBERT II


## 5. Do you have enough material for each adversarial category?
Week 3 needs examples of **negation, swapped comparative, changed unit, changed entity,
changed number**. `entity` is true for almost every question, so it is a weak filter by design.
Check that `year`/`number`/`roman_numeral`, `comparative` and `unit` have enough questions.
`yes_no` is the natural fit for negation. If it is tiny, flag it for the taxonomy session with P2.

In [12]:
feat_counts = Counter(f for s in kept.features for f in s.split(",") if f)
pd.Series(feat_counts).sort_values(ascending=False)

entity           60074
year             14966
comparative       9143
number            6033
unit              1000
roman_numeral      532
yes_no              35
dtype: int64

## 6. Dry-run the selection (preview only, nothing is saved)
Same function the script calls. The funnel numbers go into your data-source note.

In [13]:
N_QUESTIONS, SEED = 800, 42      # keep identical to the script's --n and --seed
selected, report = bsq.select(records, cfg, n=N_QUESTIONS, seed=SEED, config_name=CONFIG, split=SPLIT)
display(pd.DataFrame(report["funnel"]))
print(report["selected_feature_counts"])

,stage,remaining
0,total_rows,87622
1,after_answer_missing,87622
2,after_answer_too_long,84996
3,after_question_length,73798
4,after_time_sensitive,73186
5,after_multi_answer,72190
6,after_no_modifiable_feature,62791
7,after_duplicate_question,62625
8,sampled,800


{'entity': 765, 'number': 75, 'comparative': 125, 'year': 188, 'roman_numeral': 8, 'unit': 10}


## 7. Answer spot-check (run AFTER the script has produced `source_questions.json`)
The dataset can contain wrong or ambiguous answers. For example, in the HF preview the aliases for
"first European country to abolish capital punishment" included Norway while a search snippet in the
same row said Portugal. Verify ~30 answers **against an independent source**, not against the dataset.
Fill in the CSV, then summarise % ok / wrong / ambiguous in the data-source note.

In [21]:
import json, random
from pathlib import Path
import pandas as pd

DATA_DIR = Path(r"D:\mlse")

src = json.loads((DATA_DIR / "source_questions.json").read_text(encoding="utf-8"))
sample = random.Random(7).sample(src, 30)

check = pd.DataFrame([{
    "id": s["id"],
    "question": s["question"],
    "dataset_answer": s["ground_truth_answer"],
    "some_aliases": " | ".join(s["source_metadata"]["aliases"][:5]),
    "verified_answer": "",
    "verdict (ok/wrong/ambiguous)": "",
    "notes": "",
} for s in sample])

out = DATA_DIR / "answer_spotcheck.csv"
check.to_csv(out, index=False)
print("Wrote", out, "- fill in the blank columns, then summarise the results.")
check.head()

Wrote D:\mlse\answer_spotcheck.csv - fill in the blank columns, then summarise the results.


,id,question,dataset_answer,some_aliases,verified_answer,verdict (ok/wrong/ambiguous),notes
0,q0332,What Christian church observes The Sabbath on Saturday?,The Seventh Day Adventist Church,Seventh Day Adventists | Seventh day adventists | Adventist of the Seventh Day | Seventh-day Adventism | Seventh-Day...,,,
1,q0155,"Besides Uranium, which extremely radioactive element is extracted from Pitchblende?",RADIUM,Radium | Radium (Ra) | Ra-226 | Element 88 | RADIUM,,,
2,q0405,What is the occupation of Adam Bede in the book by George Eliot?,A CARPENTER,Housewright | Carpenter shop | A CARPENTER | Carpenter | Medieval carpentry,,,
3,q0667,Which all-girl group had a hit with 'Sound Of The Underground' in 2003?,'GIRLS ALOUD',The Sound of Girls Aloud: The Greatest Hits Volume 2 | Girls Aloud | Girls Aloud (band) | Girls' Aloud | Kleptomania...,,,
4,q0050,"MiniBus, NintendoJet, ScareBus, Chainsaw, Deathjet are all nicknames for which model of aircraft?",Airbus A320,Airbus A319-114 | Airbus A319-115 | Airbus A319-112 | Airbus A319-111 | A319ceo,,,


## Next
1. Fix any filter problems you found → rerun the script.
2. Spot-check answers (section 7) and log the results.
3. Write the data-source note (source, citation, config/split, filter rules + funnel from
   `source_questions_report.json`, copyright statement, license "unknown" on the HF card).
4. Push `data/source_questions.json` and tell P2.